In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score, accuracy_score, mean_squared_error 
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")

# Merge both datasets
data = pd.merge(ratings, movies, on="movieId")

# Average rating
avg_ratings = ratings.groupby("movieId")["rating"].mean().reset_index()

# Merge
movies = pd.merge(movies, avg_ratings, on="movieId")

# Extract year from title
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies['year'] = pd.to_numeric(movies['year'], errors='coerce')

# Clean genres
movies['genres'] = movies['genres'].str.replace('|', ' ')

In [147]:
print(ratings.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [148]:
print(data.head())

   userId  movieId  rating  timestamp                        title  \
0       1        1     4.0  964982703             Toy Story (1995)   
1       1        3     4.0  964981247      Grumpier Old Men (1995)   
2       1        6     4.0  964982224                  Heat (1995)   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0  964982931   Usual Suspects, The (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                               Comedy|Romance  
2                        Action|Crime|Thriller  
3                             Mystery|Thriller  
4                       Crime|Mystery|Thriller  


In [149]:
data.sample(6)


,userId,movieId,rating,timestamp,title,genres
16767,105,96606,5.0,1446572580,Samsara (2011),Documentary
32612,222,466,3.5,1391352616,Hot Shots! Part Deux (1993),Action|Comedy|War
76169,479,3544,1.0,1039362003,Shakes the Clown (1992),Comedy
64270,414,7323,4.0,1109974401,"Good bye, Lenin! (2003)",Comedy|Drama
74860,474,7981,4.0,1107193601,Infernal Affairs (Mou gaan dou) (2002),Crime|Drama|Thriller
95102,600,2,4.0,1237764627,Jumanji (1995),Adventure|Children|Fantasy


In [150]:
data.shape

(100836, 6)

In [151]:
data.isnull()

,userId,movieId,rating,timestamp,title,genres
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,False
3,False,False,False,False,False,False
4,False,False,False,False,False,False
...,...,...,...,...,...,...
100831,False,False,False,False,False,False
100832,False,False,False,False,False,False
100833,False,False,False,False,False,False
100834,False,False,False,False,False,False


In [152]:
data.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
title        0
genres       0
dtype: int64

In [153]:
data.duplicated()

0         False
1         False
2         False
3         False
4         False
          ...  
100831    False
100832    False
100833    False
100834    False
100835    False
Length: 100836, dtype: bool

In [154]:
data[data.duplicated()]

,userId,movieId,rating,timestamp,title,genres


In [155]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
 4   title      100836 non-null  str    
 5   genres     100836 non-null  str    
dtypes: float64(1), int64(3), str(2)
memory usage: 4.6 MB


In [156]:
data.describe()

,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


In [157]:
data['rating'].value_counts()

rating
4.0    26818
3.0    20047
5.0    13211
3.5    13136
4.5     8551
2.0     7551
2.5     5550
1.0     2811
1.5     1791
0.5     1370
Name: count, dtype: int64

In [158]:
data['genres'].value_counts()

genres
Comedy                        7196
Drama                         6291
Comedy|Romance                3967
Comedy|Drama|Romance          3000
Comedy|Drama                  2851
                              ... 
Children|Crime|Drama             1
Drama|Fantasy|Thriller|War       1
Drama|Horror|Romance             1
Horror|Romance|Sci-Fi            1
Action|Crime|Drama|Sci-Fi        1
Name: count, Length: 951, dtype: int64

In [159]:
data['userId'].value_counts()

userId
414    2698
599    2478
474    2108
448    1864
274    1346
       ... 
431      20
442      20
569      20
576      20
595      20
Name: count, Length: 610, dtype: int64

In [160]:
movies.head()

,movieId,title,genres,rating,year
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,3.920930,1995.0
1,2,Jumanji (1995),Adventure Children Fantasy,3.431818,1995.0
2,3,Grumpier Old Men (1995),Comedy Romance,3.259615,1995.0
3,4,Waiting to Exhale (1995),Comedy Drama Romance,2.357143,1995.0
4,5,Father of the Bride Part II (1995),Comedy,3.071429,1995.0


In [161]:
movies.tail()

,movieId,title,genres,rating,year
9719,193581,Black Butler: Book of the Atlantic (2017),Action Animation Comedy Fantasy,4.0,2017.0
9720,193583,No Game No Life: Zero (2017),Animation Comedy Fantasy,3.5,2017.0
9721,193585,Flint (2017),Drama,3.5,2017.0
9722,193587,Bungo Stray Dogs: Dead Apple (2018),Action Animation,3.5,2018.0
9723,193609,Andrew Dice Clay: Dice Rules (1991),Comedy,4.0,1991.0


In [162]:
print(movies.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres    rating    year  
0  Adventure Animation Children Comedy Fantasy  3.920930  1995.0  
1                   Adventure Children Fantasy  3.431818  1995.0  
2                               Comedy Romance  3.259615  1995.0  
3                         Comedy Drama Romance  2.357143  1995.0  
4                                       Comedy  3.071429  1995.0  


In [163]:
movies.sample(10)

,movieId,title,genres,rating,year
5428,25996,"Star Is Born, A (1954)",Drama Musical,3.50,1954.0
752,987,Bliss (1997),Drama Romance,3.00,1997.0
8641,121007,Space Buddies (2009),Adventure Children Fantasy Sci-Fi,4.00,2009.0
5425,25959,Annie Get Your Gun (1950),Comedy Musical Romance Western,2.50,1950.0
5004,7767,"Best of Youth, The (La meglio gioventù) (2003)",Drama,4.75,2003.0
4482,6636,"Sure Thing, The (1985)",Comedy Romance,3.75,1985.0
9523,172591,The Godfather Trilogy: 1972-1990 (1992),(no genres listed),4.75,1992.0
6595,55814,"Diving Bell and the Butterfly, The (Scaphandre...",Drama,4.10,2007.0
6160,44773,"Dead Hate the Living!, The (2000)",Comedy Horror,3.00,2000.0
8744,128542,Wyrmwood (2015),Action Horror Sci-Fi,3.50,2015.0


In [164]:
movies.shape

(9724, 5)

In [165]:
movies.isnull()

,movieId,title,genres,rating,year
0,False,False,False,False,False
1,False,False,False,False,False
2,False,False,False,False,False
3,False,False,False,False,False
4,False,False,False,False,False
...,...,...,...,...,...
9719,False,False,False,False,False
9720,False,False,False,False,False
9721,False,False,False,False,False
9722,False,False,False,False,False


In [166]:
movies.isnull().sum()

movieId     0
title       0
genres      0
rating      0
year       13
dtype: int64

In [167]:
movies.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
9719    False
9720    False
9721    False
9722    False
9723    False
Length: 9724, dtype: bool

In [168]:
movies.duplicated().sum()

np.int64(0)

In [169]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 9724 entries, 0 to 9723
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9724 non-null   int64  
 1   title    9724 non-null   str    
 2   genres   9724 non-null   str    
 3   rating   9724 non-null   float64
 4   year     9711 non-null   float64
dtypes: float64(2), int64(1), str(2)
memory usage: 380.0 KB


In [170]:
movies.describe()

,movieId,rating,year
count,9724.000000,9724.000000,9711.000000
mean,42245.024373,3.262448,1994.659973
std,52191.137320,0.869874,18.492137
min,1.000000,0.500000,1902.000000
25%,3245.500000,2.800000,1988.000000
50%,7300.000000,3.416667,1999.000000
75%,76739.250000,3.911765,2008.000000
max,193609.000000,5.000000,2018.000000


In [171]:
# Get unique genres
all_genres = set()

for g in movies['genres']:
    for genre in g.split():
        all_genres.add(genre)

print("\n🎭 Available Genres:\n")
print(", ".join(sorted(all_genres)))


🎭 Available Genres:

(no, Action, Adventure, Animation, Children, Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, Horror, IMAX, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western, genres, listed)


In [172]:
cv = CountVectorizer()
genre_matrix = cv.fit_transform(movies['genres'])

similarity = cosine_similarity(genre_matrix)

In [173]:
def recommend_movies(user_genre, start_year, end_year, top_n=10, low_n=2):
    
    # Convert input genre
    user_vec = cv.transform([user_genre])
    
    scores = cosine_similarity(user_vec, genre_matrix).flatten()
    movies['score'] = scores
    
    # Filter by year
    filtered = movies[
        (movies['year'] >= start_year) &
        (movies['year'] <= end_year) &
        (movies['rating'].notna())
    ]
    
    # Top movies
    top_movies = filtered.sort_values(
        by=['score', 'rating'], ascending=False
    ).head(top_n)
    
    # Poor movies
    poor_movies = filtered.sort_values(
        by=['score', 'rating'], ascending=[False, True]
    ).head(low_n)
    
    return top_movies, poor_movies

In [174]:
# Extract all unique genres
all_genres = set()

for g in movies['genres']:
    for genre in g.split():
        if genre != '(no':  # safety for bad parsing
            all_genres.add(genre)

# Sort genres
all_genres = sorted(all_genres)

# Display nicely
print("\n🎭 Available Genres:\n")

for i, genre in enumerate(all_genres, 1):
    print(f"{i}. {genre}")


🎭 Available Genres:

1. Action
2. Adventure
3. Animation
4. Children
5. Comedy
6. Crime
7. Documentary
8. Drama
9. Fantasy
10. Film-Noir
11. Horror
12. IMAX
13. Musical
14. Mystery
15. Romance
16. Sci-Fi
17. Thriller
18. War
19. Western
20. genres
21. listed)


In [175]:
print("\n👉 Enter your preferred genres (space separated):")
user_genre = input("Genres: ")

print("\n👉 Enter year range:")
start_year = int(input("Start Year: "))
end_year = int(input("End Year: "))

top_movies, poor_movies = recommend_movies(user_genre, start_year, end_year)

print("\n🎬 Top Recommended Movies:\n")
for _, row in top_movies.iterrows():
    print(f"🎥 {row['title']} ({int(row['year'])})")
    print(f"⭐ Rating: {round(row['rating'],2)}")
    
    genres_list = row['genres'].split()
    print("🎭 Genres:", ", ".join(genres_list))
    
    print("-" * 40)

print("\n⚠️ Low Rated But Similar Movies:\n")
for _, row in poor_movies.iterrows():
    print(f"🎥 {row['title']} ({int(row['year'])})")
    print(f"⭐ Rating: {round(row['rating'],2)}")
    
    genres_list = row['genres'].split()
    print("🎭 Genres:", ", ".join(genres_list))
    
    print("-" * 40)


👉 Enter your preferred genres (space separated):


Genres:  war



👉 Enter year range:


Start Year:  2010
End Year:  2020



🎬 Top Recommended Movies:

🎥 Flowers of War, The (Jin líng shí san chai) (2011) (2011)
⭐ Rating: 4.5
🎭 Genres: Drama, War
----------------------------------------
🎥 Lincoln (2012) (2012)
⭐ Rating: 4.5
🎭 Genres: Drama, War
----------------------------------------
🎥 Generation War (2013) (2013)
⭐ Rating: 4.0
🎭 Genres: Drama, War
----------------------------------------
🎥 Virunga (2014) (2014)
⭐ Rating: 4.0
🎭 Genres: Documentary, War
----------------------------------------
🎥 Hacksaw Ridge (2016) (2016)
⭐ Rating: 3.94
🎭 Genres: Drama, War
----------------------------------------
🎥 Darkest Hour (2017) (2017)
⭐ Rating: 3.83
🎭 Genres: Drama, War
----------------------------------------
🎥 American Sniper (2014) (2014)
⭐ Rating: 3.8
🎭 Genres: Action, War
----------------------------------------
🎥 Whiskey Tango Foxtrot (2016) (2016)
⭐ Rating: 3.33
🎭 Genres: Comedy, War
----------------------------------------
🎥 Restrepo (2010) (2010)
⭐ Rating: 3.17
🎭 Genres: Documentary, War
------------------